In [ ]:
#!pip install findpeaks scikit-image

In [2]:
from utils import *
# from tonotopy import *
import findpeaks
from skimage import measure
import os

t_pre = 0.5#0.2
t_post = 0.50#0.300
bin_width = 0.005
# Créer les bins de temps"
psth_bins = np.arange(-t_pre, t_post, bin_width)

In [3]:
path = r'\\129.199.81.18\data5\eTheremin\SKIEUR\SKIEUR_20260414_SESSION_01'


In [4]:
t_pre = 0.3#0.2
t_post = 0.30#0.300
bin_width = 0.005
# Créer les bins de temps"
psth_bins = np.arange(-t_pre, t_post, bin_width)

In [5]:
data = np.load(path+r'\headstage_0\data_0.005.npy', allow_pickle=True)
features = np.load(path+r'\headstage_0\features_0.005.npy', allow_pickle=True)
gc = np.load(path+r'\headstage_0\good_clusters.npy', allow_pickle=True)
#gc = np.arange(32)



In [ ]:
bd = np.load(r'\\129.199.81.18\data6\eTheremin\ALTAI\ALTAI_20240822_SESSION_00\heatmap_bandwidth.npy')

In [ ]:
bd

In [12]:
tones = get_played_frequency(features, t_pre, t_post, bin_width, 'tracking')
# prendre les valeurs uniques de tones
unique_tones = np.load(path+r'\headstage_0\unique_tones.npy', allow_pickle=True)
unique_tones = sorted(np.unique(tones))

In [6]:
psth_check = get_psth(data, features, t_pre, t_post, bin_width, gc, 'tracking')
print("Number of neurons:", len(psth_check))
print("Number of trials for neuron 0:", len(psth_check[0]))
if len(psth_check[0]) > 0:
    print("First trial shape:", psth_check[0][0].shape)
    print("First trial values:", psth_check[0][0][:10])
else:
    print("No trials found! Check Condition and Frequency_changes fields.")

Number of neurons: 11
Number of trials for neuron 0: 2897
First trial shape: (120,)
First trial values: [0 1 0 0 0 0 0 0 0 0]


In [9]:
def get_tonotopy_t(data, features, t_pre, t_post, bin_width, good_clusters, unique_tones, max_freq, min_freq, condition, save_name):
    """""
    
    Fonction qui pour une session renvoie
    les heatmaps (psth x freq) pour la tonotopie mais ne les plot pas
    une heatmap par neurone
    uniquement les good_clusters
    attention : les heatmaps sont brutes (pas de traitements, ni smoothed... etc)
    
    input : data, features, t_pre, t_post (pour le psth), bins, good_clusters et condition ("tracking" ou "playback)
            unique_tones : ce sont les tons uniques qui ont été joués pendant la session (33 en tout)
            max_freq, min_freq : indices min et max des fréquences extrêmes à partir desquelles on ne prend pas les psth pour les heatmap
            (car pas assez de présentations donc ca déconne) min_freq = 5, max_freq = 7
            condition : 'tracking' ou 'playback
    ouput : 1 tableau contenant 1 heatmap par good_cluster 
            heatmap non smoothée
    """
    
    #je prends les psth de chaque neurones et la fréquence associée à chaque psth
    psth = get_psth(data, features, t_pre, t_post, bin_width, good_clusters, condition)
    tones = get_played_frequency(features, t_pre, t_post, bin_width, condition)
    tones = [int(x) for x in tones]
    unique_tones = [int(x) for x in unique_tones]
    
    psth_bins = np.arange(-t_pre, t_post + bin_width, bin_width)
    
    n_clus = len(good_clusters)
     

    tones = np.array(tones)
    tones = [int(x) for x in tones]
    unique_tones_test = np.unique(tones)
    unique_tones = [int(x) for x in unique_tones]
    
    heatmaps = []

    for c, clus in enumerate(good_clusters):  
        clus_psth = np.array(psth[c])
        print(len(clus_psth))
        average_psth_list = []
        
        for tone in unique_tones:
            mask = (tones == tone)
            if len(clus_psth[mask])>0: 
                print('ok')
                average_psth = np.mean(clus_psth[mask], axis=0)
                average_psth_list.append(average_psth)
                print(average_psth)
            else:
                print('shit')
                average_psth_list.append(np.zeros_like(psth_bins[:-1]))
    
        average_psths_array = np.array(average_psth_list)
        
        t_0 = int(t_pre/bin_width)
        # faire la moyenne sur toute la heatmap
        #mu = np.nanmean(average_psths_array[:][0:t_0], axis=0)
        #mu = np.nanmean(mu, axis=0)
        
        #je retire la moyenne de la heatmap avant le stim
        #trouver le bin du stim
        
        
        #heatmap = average_psths_array[min_freq:-max_freq]-mu
        #heatmap = average_psths_array-mu
        heatmap = average_psths_array 
        heatmaps.append(heatmap)
        #np.save(save_name, np.array(heatmaps))
    
    return heatmaps

In [13]:

tones = get_played_frequency(features, t_pre, t_post, bin_width, 'tracking')
tones = [int(x) for x in tones]
unique_tones = [int(x) for x in unique_tones]
    
psth_bins = np.arange(-t_pre, t_post + bin_width, bin_width)
    
n_clus = len(gc)
     

tones = np.array(tones)
tones = [int(x) for x in tones]
unique_tones = [int(x) for x in unique_tones]
for tone in unique_tones:
    mask = (tones == tone)
    print(tones[mask])


2000
2000
2000
2000
2000
2000
2000
2000
2000
2000
2000
2000
2000
2000
2000
2000
2000
2000
2000


In [17]:
unique_tones

[460,
 533,
 617,
 715,
 828,
 959,
 1111,
 1287,
 1490,
 1727,
 2000,
 2317,
 2684,
 3109,
 3601,
 4172,
 4832,
 5598,
 6484]

In [15]:
heatmaps = get_tonotopy_t(data, features, t_pre, t_post, bin_width, gc, unique_tones, 0, 0, 'tracking', 'heatmaps')


2897
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
2897
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
2897
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
2897
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
2897
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
2897
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
2897
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
2897
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
2897
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
2897
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit
shit


In [16]:
hm = heatmaps[0]  # 第一个神经元的热图
print("Shape:", hm.shape)                     # 应为 (n_tones, 120)
print("Min:", np.nanmin(hm))
print("Max:", np.nanmax(hm))
print("Mean:", np.nanmean(hm))
print("Std:", np.nanstd(hm))
print("1st percentile:", np.percentile(hm, 1))
print("99th percentile:", np.percentile(hm, 99))

Shape: (19, 120)
Min: 0.0
Max: 0.0
Mean: 0.0
Std: 0.0
1st percentile: 0.0
99th percentile: 0.0


In [ ]:
heatmaps

In [ ]:
if not os.path.exists(path + 'heatmap_plot_playback.npy'):
    print("calculating heatmaps")
    heatmaps = get_tonotopy(data, features, t_pre, t_post, bin_width, gc, unique_tones, 0, 0, 'playback', 'heatmaps')

else:
    heatmaps = np.load(path + 'heatmap_plot_playback.npy', allow_pickle = True)
    print('heatmaps already exist')

#récupérer les heatmaps
plot_heatmap_bandwidth(heatmaps,3, gc,unique_tones, 2, 2, bin_width, psth_bins, t_pre,path, '', 'playback')

In [ ]:
heatmaps = get_tonotopy(data, features, t_pre, t_post, bin_width, gc, unique_tones, 0, 0, 'tracking', 'heatmaps')


In [ ]:
tones = get_played_frequency(features, t_pre, t_post, bin_width, 'tracking')

In [ ]:

def get_tonotopy(data, features, t_pre, t_post, bin_width, good_clusters, unique_tones, max_freq, min_freq, condition, save_name):
    """""
    
    Fonction qui pour une session renvoie
    les heatmaps (psth x freq) pour la tonotopie mais ne les plot pas
    une heatmap par neurone
    uniquement les good_clusters
    attention : les heatmaps sont brutes (pas de traitements, ni smoothed... etc)
    
    input : data, features, t_pre, t_post (pour le psth), bins, good_clusters et condition ("tracking" ou "playback)
            unique_tones : ce sont les tons uniques qui ont été joués pendant la session (33 en tout)
            max_freq, min_freq : indices min et max des fréquences extrêmes à partir desquelles on ne prend pas les psth pour les heatmap
            (car pas assez de présentations donc ca déconne) min_freq = 5, max_freq = 7
            condition : 'tracking' ou 'playback
    ouput : 1 tableau contenant 1 heatmap par good_cluster 
            heatmap non smoothée
    """
    
    #je prends les psth de chaque neurones et la fréquence associée à chaque psth
    psth = get_psth(data, features, t_pre, t_post, bin_width, good_clusters, condition)
    tones = get_played_frequency(features, t_pre, t_post, bin_width, condition)
    tones = [int(x) for x in tones]
    unique_tones = [int(x) for x in unique_tones]
    
    psth_bins = np.arange(-t_pre, t_post + bin_width, bin_width)
    
    n_clus = len(good_clusters)
     

    tones = np.array(tones)
    tones = [int(x) for x in tones]
    unique_tones_test = np.unique(tones)
    unique_tones = np.array([int(x) for x in unique_tones])

    heatmaps = []

    for c, clus in enumerate(good_clusters):  
        clus_psth = np.array(psth[c])
        average_psth_list = []
        
        for tone in unique_tones:

            mask = (tones == tone)
            if len(clus_psth[mask])>0: #au moins 20 présentations d'une fréquence
                average_psth = np.mean(clus_psth[mask], axis=0)
                average_psth_list.append(average_psth)
                print(average_psth)
            else:
                average_psth_list.append(np.zeros_like(psth_bins[:-1]))
    
        average_psths_array = np.array(average_psth_list)
        
        t_0 = int(t_pre/bin_width)
        # faire la moyenne sur toute la heatmap
        #mu = np.nanmean(average_psths_array[:][0:t_0], axis=0)
        #mu = np.nanmean(mu, axis=0)
        
        #je retire la moyenne de la heatmap avant le stim
        #trouver le bin du stim
        
        
        #heatmap = average_psths_array[min_freq:-max_freq]-mu
        #heatmap = average_psths_array-mu
        heatmap = average_psths_array 
        heatmaps.append(heatmap)
    
    return heatmaps

In [ ]:
hmm =  get_tonotopy(data, features, t_pre, t_post, bin_width, gc, unique_tones, 0, 0, 'tracking', "")

In [ ]:
psth = get_psth(data, features, t_pre, t_post, bin_width, gc, 'tracking')
tones = get_played_frequency(features, t_pre, t_post, bin_width, 'tracking')
tones = [int(x) for x in tones]
unique_tones = [int(x) for x in unique_tones]

In [ ]:
tones

In [ ]:
unique_tones

In [ ]:
features

In [ ]:
heatmaps